# 0. Imports

In [1]:
#!pip install -qq ipython numpy pandas scikit-learn statsmodels xgboost torch

In [2]:
import sys
import warnings
from pathlib import Path
from IPython.display import display

import numpy as np
import pandas as pd

import sklearn
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, root_mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline


In [3]:
!python --version
print("numpy", np.__version__)
print("pandas", pd.__version__)
print("scikit-learn", sklearn.__version__)

Python 3.10.18
numpy 2.2.6
pandas 2.3.3
scikit-learn 1.7.2


# 1. Preprocessing

In [4]:
# AUTOREGRESSIVE (LAG) FEATURES

lags = sorted(set(
    list(range(1, 7)) + [12, 18]
    + list(range(24, 27)) + [36, 48]
    + [24*i for i in range(3,7)]
    + [24*7*i for i in range(1,5)]
))
lagFeatures = [f"Adjusted demand -{h} hr" for h in lags]

In [5]:
# CALENDAR FEATURES

# Raw integer calendar features
intDateTimeFeatures = ["Hour", "Month", "DayOfWeek", "DayOfYear"]

# Low order hour of day and day of year Fourier term features
hourFourierFeatures, dayFourierFeatures = [], []
for i in (1, 2, 3):
    argStr = (f"{i}*" if i>1 else "") + "Hour"
    hourFourierFeatures.extend([f"sin({argStr})", f"cos({argStr})"])
    argStr = (f"{i}*" if i>1 else "") + "DayOfYear"
    dayFourierFeatures.extend([f"sin({argStr})", f"cos({argStr})"])

# Hour of day and day of week one-hot encodings.
# Weekend and holiday flags.
hourDummyFeatures = [f"Hour_Flag_{h}" for h in range(24)]
dayDummyFeatures = ["Day_Flag_Weekend", "Day_Flag_Holiday"]
dayDummyFeatures += [f"DayOfWeek_Flag_{d}" for d in range(7)]
monthDummyFeatures = [f"Month_Flag_{m}" for m in range(1, 13)]

# Collate all calendar features
calendarFeatures = (
    intDateTimeFeatures
    + hourFourierFeatures
    + dayFourierFeatures
    + hourDummyFeatures
    + dayDummyFeatures
    + monthDummyFeatures
)


In [6]:
# ENERGY FEATURES

energyFeatures = [
    "Adjusted net generation",
    "Adjusted total interchange",
    "FPC", "FMPP", "SOCO", "TEC",
    "JEA", "SEC", "HST", "GVL",
]


In [7]:
# WEATHER FEATURES

skyCodes = ['BKN', 'CLR', 'FEW', 'SCT', 'OVC', 'NA']
directions = ["N", "NE", "E", "SE", "S", "SW", "W", "NW", "VRB"]

weatherFeatures = ([
    "HourlyDryBulbTemperature",
    "HourlyPrecipitation",
    "HourlyRelativeHumidity",
    "HourlySeaLevelPressure",
    "HourlyVisibility",
    "HourlyWindSpeed"]
    + [f"HourlySkyConditions_Flag_{code}" for code in skyCodes]
    + [f"HourlyWindDirection_Flag_{d}" for d in directions]
    + ["sin(HourlyWindDirection)", "cos(HourlyWindDirection)"]
)


In [8]:
# Define other convenient feature variables.

target = "Adjusted demand"
allFeatures = (
    ['t', target]
    + lagFeatures 
    + calendarFeatures 
    + energyFeatures 
    + weatherFeatures
)
someFeatures = (
    ['t', target]
    + lagFeatures[:1] 
    + hourFourierFeatures[:2] + dayFourierFeatures[:2] + dayDummyFeatures[:1]
    + energyFeatures[:2] 
    + weatherFeatures[:1]
)


### 1.3 Data Splits

In [9]:
# reproducibility
np.random.seed(0)

# --- paths ---
data_root   = Path("data/clean")
train_dir   = data_root / "train"
val_dir     = data_root / "val"

DFtrain = pd.read_pickle(train_dir / "DFtrain.pkl")
DFval   = pd.read_pickle(val_dir   / "DFval.pkl")

# --- define target and features (ADJUST target name to yours) ---
TARGET_COL   = "Adjusted demand"   # <--- change to your target column
TIME_COL     = "t"        # if you have a time column
FEATURE_COLS = someFeatures[2:]

print("n_train:", len(DFtrain), "n_val:", len(DFval))
print("n_features:", len(FEATURE_COLS))

n_train: 59658 n_val: 12784
n_features: 9


In [10]:
# --- ARX (Ridge) hyperparameter tuning on train/val ---

def eval_ridge(alpha, X_train, y_train, X_val, y_val):
    """Fit a ridge pipeline for a given alpha and return validation RMSE."""
    model = Pipeline([
        ("scaler", StandardScaler()),
        ("ridge", Ridge(alpha=alpha))
    ])
    model.fit(X_train, y_train)
    y_val_pred = model.predict(X_val)
    rmse = root_mean_squared_error(y_val, y_val_pred)
    return rmse, model

X_train_ARX = DFtrain[FEATURE_COLS].to_numpy()
y_train_ARX = DFtrain[TARGET_COL].to_numpy()
X_val_ARX   = DFval[FEATURE_COLS].to_numpy()
y_val_ARX   = DFval[TARGET_COL].to_numpy()

alpha_grid = [1e-3, 1e-2, 1e-1, 1, 10, 100]

best_alpha = None
best_rmse  = np.inf
best_ARX   = None

for alpha in alpha_grid:
    rmse, model = eval_ridge(alpha, X_train_ARX, y_train_ARX, X_val_ARX, y_val_ARX)
    print(f"ARX alpha={alpha:6g}  val RMSE={rmse:.4f}")
    if rmse < best_rmse:
        best_rmse  = rmse
        best_alpha = alpha
        best_ARX   = model

print(f"\nBest ARX alpha={best_alpha}  (val RMSE={best_rmse:.4f})")

ARX alpha= 0.001  val RMSE=239.5122
ARX alpha=  0.01  val RMSE=239.5123
ARX alpha=   0.1  val RMSE=239.5133
ARX alpha=     1  val RMSE=239.5233
ARX alpha=    10  val RMSE=239.6247
ARX alpha=   100  val RMSE=240.7527

Best ARX alpha=0.001  (val RMSE=239.5122)
